# Ten Trails — MF6 model from the USG export

Picks up where `layer-stack.ipynb` leaves off. Everything comes out of
`from_usg/` and goes in through the normal package-first API.

In [ ]:
import myflopy as mf
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np

CRS = 2927
usg = Path(r'/home/lukem/models/mf6/tentrails/from_usg')
arrays = usg / 'arrays'
bcs = usg / 'boundaries.gpkg'

periods = pd.read_csv(usg / 'periods.csv')
nper = len(periods)
nper

## Grid and layers

Same as `layer-stack.ipynb`. `attach_to_grid()` publishes the surfaces as numbered
columns, which is the form `mf.sfr` reads.

In [ ]:
# vor, layer_stack from layer-stack.ipynb

disv = layer_stack.to_disv(vor)
res = layer_stack.build(vor)
res.attach_to_grid()

ctx = mf.ModelContext(
    grid=vor,
    domain=res.idomain,
    dates=pd.date_range('2017-09-30', periods=nper, freq='ME'),
)
res.units

## Properties

The old model's five layers line up with yours one for one — its Qpog1/Qpog2 pair is
your `Qpog` split 50/50.

In [ ]:
def arr(name, fill='median'):
    """One exported raster on the grid, with cells outside its coverage filled.

    Your domain is larger than the old model's, so 2,287 cells have no source
    value. A single NaN makes the whole solution NaN, so they must be filled --
    MF6 does not warn, it just returns NaN for every budget term.
    """
    a = mf.Surface.raster(arrays / f'{name}.tif').values(vor)
    gap = np.isnan(a)
    if gap.any():
        default = np.nanmedian(a) if isinstance(fill, str) else fill
        a = np.where(gap, default, a)
    return a


def per_layer(prefix, fill='median'):
    return [arr(f'{prefix}_{lay}', fill) for lay in range(1, res.nlay + 1)]


outside = np.isnan(mf.Surface.raster(arrays / 'k_1.tif').values(vor))
print(f'{outside.sum():,} of {vor.ncpl:,} cells lie outside the old model; filled')


npf = mf.npf(k=per_layer('k'), k33=per_layer('k33'), icelltype=1)
sto = mf.sto(ss=per_layer('ss'), sy=per_layer('sy'), iconvert=1, transient={0: True})
# starting heads: where the old model had none, start at ground
ic = mf.ic(strt=per_layer('strt', fill=np.asarray(res.top).ravel()))

## Boundaries

`mf.Spread` on the conductance — otherwise each feature's value is copied to every
cell it crosses and the drain gets 5.6x stronger on this grid than it was on the old
one. Heads are copied, which is right.

Drain inverts come across as absolute elevations and a few land below their new cell
bottom, which MF6 rejects outright. `clamp_into_cell` raises those onto the cell
floor and says how many it moved — close, not exact, which is the deal with any
re-gridding. (`elevation_below_top` is also in the file if you would rather place
them at a fixed depth below ground; on this stack that fits worse, not better.)

In [ ]:
def bc(package, layer, **fields):
    return getattr(mf, package).gpkg(
        bcs, layer=layer, context=ctx, nper=nper,
        layer_field='layer', layer_base=1, name_field='name', **fields,
    )


drn = bc('drn', 'drn_lines', elevation='elevation',
         conductance=mf.Spread('conductance'))


def clamp_into_cell(spec, builder, label, margin=0.5):
    """Raise any elevation/head sitting below its cell bottom; MF6 rejects those."""
    botm = np.asarray(res.botm)
    data, moved = {}, set()
    for period, rows in spec.options['stress_period_data'].items():
        out = []
        for cellid, value, *rest in rows:
            floor = botm[cellid[0]][cellid[1]] + margin
            if value < floor:
                value = floor
                moved.add(cellid)
            out.append([cellid, value, *rest])
        data[period] = out
    print(f'{label}: {len(moved)} of {len(data[0])} raised onto the cell floor')
    return builder(stress_period_data=data, boundnames=True)


drn = clamp_into_cell(drn, mf.drn, 'drn')
ghb = bc('ghb', 'ghb_lines', head='head',
         conductance=mf.Spread('conductance'))
ghb = clamp_into_cell(ghb, mf.ghb, 'ghb')

# CHD from the POINT layer: two constant heads in one cell is an MF6 error, and the
# lines put 11 pairs together on this grid. Points resolve to one cell each.
chd = bc('chd', 'chd', head=[f'head_m{(p % 12) + 1:02d}' for p in range(nper)])

for name, spec in (('drn', drn), ('ghb', ghb), ('chd', chd)):
    recs = spec.options['stress_period_data'][0]
    print(f'{name}  {len(recs):5d} records  {len({r[0][1] for r in recs}):5d} cells')

## Streams and lakes

`rbth` and `man` are made up — CLN has no equivalent. `reach_top=None` samples your
top and clamps it monotone downstream, which is what the old model did.

Two things about the streambed. `rbth` is 0.05 ft rather than the exported 1.0,
because MF6 requires `rtp - rbth` above the cell bottom and the pinch floor makes
layer 1 exactly 0.1 ft thick under 99 of the 652 stream cells. And `reach_top` is
given explicitly as the model top: `reach_top=None` samples the top and then clamps
it monotone downstream, which on a surface with rises drags the bed *below* the cell
(measured: 10 reaches, worst 2.6 ft under). Conditioning the profile is your step —
this just puts the bed on the top and keeps it inside its cell.

Lakes are flat-bottomed: `connection_modes='rectangular'` spans `lake_bottom` to
`lake_top`, and the CLN carries no bathymetry so there is nothing better to use.
No lake tables — a table gives a stage/volume/area curve, and a flat-bottomed lake
is a prism whose area MF6 already gets from the cell footprint.

In [ ]:
streams = gpd.read_file(usg / 'streams.gpkg', layer='streams')

RBTH = 0.05

# streambed = the top of whichever layer the reach lands in (SFR uses top_active,
# which is not layer 1 where layer 1 is pinched out), floored inside that cell
top = np.asarray(res.top).ravel()
botm = np.asarray(res.botm)
idom = np.asarray(res.idomain)
first = (idom > 0).argmax(axis=0)

on_stream = vor.gdf_vorPolys.index[vor.gdf_vorPolys.intersects(streams.union_all())]
reach_top = {}
for c in on_stream:
    lay = int(first[c])
    cell_top = top[c] if lay == 0 else botm[lay - 1][c]
    reach_top[(lay, int(c))] = max(cell_top, botm[lay][c] + RBTH + 0.01)

sfr = mf.sfr(
    context=ctx, nper=nper, streams=streams, stream_id='stream_id',
    width='rwid', streambed_k='rhk', gradient='rgrd',
    streambed_thickness=RBTH, roughness='man', reach_top=reach_top,
    min_reach_length=1.0,   # corner clips; MF6 divides by reach length
)
streams[['stream_id', 'rwid', 'rhk', 'rgrd']]

In [ ]:
lakes = gpd.read_file(usg / 'lakes.gpkg', layer='lakes')
forcing = pd.read_csv(usg / 'lake_forcing.csv')

LAKE_DEPTH = 10.0   # ft; the exported bed IS the old model top, so pick a depth
lakes['flat_bottom_elev'] = lakes['lake_bottom'] - LAKE_DEPTH

lak = mf.lak(
    context=ctx, nper=nper, lakes=lakes, lake_id_field='lake_id',
    starting_stage='strt', bed_leakance='bedleak',
    connection_modes='rectangular',
    lake_bottom='flat_bottom_elev',
    lake_top='lake_bottom',
    rainfall={p: dict(g.set_index('lake_id')['rainfall'])
              for p, g in forcing.groupby('period')},
    evaporation={p: dict(g.set_index('lake_id')['evaporation'])
                 for p, g in forcing.groupby('period')},
)
lakes[['lake_id', 'area_ft2', 'strt', 'flat_bottom_elev', 'bedleak']]

## Recharge and ET

72 periods, 11 distinct recharge arrays — `periods.csv` maps them.

In [ ]:
rch = mf.rch.array(
    context=ctx,
    recharge={int(r.period): arr(f'recharge_{int(r.recharge_array):02d}')
              for r in periods.itertuples()},
)

evt = mf.evt.array(
    context=ctx,
    surface=arr('et_surface_01'),
    rate={int(r.period): arr(f'et_rate_{int(r.et_rate_array):02d}')
          for r in periods.itertuples()},
    depth=arr('et_depth_01'),
)

## Simulation

In [ ]:
model = mf.gwf(
    'tentrails',
    context=ctx,
    newtonoptions='NEWTON UNDER_RELAXATION',
    packages=[disv, npf, sto, ic, drn, ghb, chd, sfr, lak, rch, evt,
              mf.oc(head_filerecord='tentrails.hds',
                    budget_filerecord='tentrails.cbb',
                    saverecord=[('HEAD', 'ALL'), ('BUDGET', 'ALL')])],
)

sim = mf.simulation(
    model,
    name='tentrails',
    tdis=mf.tdis(
        nper=nper,   # mf.tdis defaults to 1 and does NOT infer it from perioddata
        perioddata=[(r.perlen, int(r.nstp), r.tsmult) for r in periods.itertuples()],
        time_units='days',
        start_date_time='2017-09-30',
    ),
    solver=mf.ims(models=['tentrails'], complexity='COMPLEX'),
)

In [ ]:
proj = mf.Project(Path(r'/home/lukem/models/mf6/tentrails'), name='tentrails')
run = proj.prepare_run('from_usg', sim)
run.execute()

---

Worth knowing:

- Old layer 5 (`Qpon`) carries the only deep GHB — 53 records at `head = 50 ft`.
  It lands in your `Qponf`.
- The old ETS was segmented; `mf.evt.array` drops the segments and applies a
  straight linear decline. `mf.evt(...)` in list form keeps them, at 72 x 9,090 records.
- CHD is a 12-month cycle. `head='head'` instead of the monthly list gives one
  value for the whole run.
- `manifest.json` and `README.md` in `from_usg/` list every column and its units.